In [ ]:
# Env
NUM_AGENTS = 2
HEIGHT = 6
WIDTH = 6
SPAWN_PROB_PER_CELL = 0.05  # 0.05
DESPAWN_PROB_PER_CELL = 0.09
DISCOUNT = 0.99 


NUM_MONTE_CARLO_SAMPLES = 1000  # Number of full episodes to run PER test state for averaging
EPISODE_LENGTH = 1000           # How many steps to simulate into the future for each episode


In [2]:
import sys
sys.path.append("../..")
import numpy as np
from orchard.environment import MoveAction
from tqdm import tqdm
import pickle

from tadd_helpers.env_functions import *
from tadd_helpers.eval_functions import *

In [3]:
def update_move_state(s: State, agent_idx: int, move_vector: np.ndarray):
    """
    Simulates an agent taking an action. Does not modify in place.
    
    Returns:
        tuple: (next_state: State, reward: float)
    """
    old_agent_pos = s.agent_position(agent_idx)
    new_agent_pos = np.clip(old_agent_pos + move_vector, [0, 0], [s.H - 1, s.L - 1]) 
    
    s_new = s.copy()
    s_new.set_agent_position(agent_idx, new_agent_pos)

    reward = 0
    return s_new, reward
    

In [4]:
def update_spawn_despawn_and_pick(s: State, agent_idx: int, picked: bool):
    """Simulate the second phase. Agents do not move in this phase but apples may change."""
    reward = 0
    s_new = s.copy()
    if picked:
        picker_pos = s_new.agent_position(agent_idx)
        assert s_new.apples[tuple(picker_pos)] > 0 # must be that an agent picked up apple if picked is True
        s_new.remove_apple_at(picker_pos)
        reward = 1
    spawn_apples(s_new, SPAWN_PROB_PER_CELL)
    despawn_apples(s_new, DESPAWN_PROB_PER_CELL)
    return s_new, reward

In [5]:
# Load the pre-generated states that we want to find the true value of
states_to_evaluate: list[State] = []
with open("test_states.pkl", "rb") as f:
    states_to_evaluate = pickle.load(f)

print(f"Loaded {len(states_to_evaluate)} states to evaluate.")
for state in states_to_evaluate:
    print(f"  - {state.name}")

Loaded 3 states to evaluate.
  - State after 20 steps
  - State after 500 steps
  - State after 999 steps


### Get CNN Centralized Estimate Value

In [6]:
# --- Monte Carlo Simulation Loop ---

# This dictionary will store the results
true_value_results = {}

# 1. Outer Loop: Iterate through each state we want to evaluate
for start_state in states_to_evaluate:
    print(f"\\n--- Running Monte Carlo Simulation for state: '{start_state.name}' ---")
    
    value_samples = [] # Store the discounted reward sum from each full episode

    # 2. Averaging Loop: Run many episodes to get a stable average value
    for i in tqdm(range(NUM_MONTE_CARLO_SAMPLES), desc="Simulating Episodes"):
        
        # --- Initialize a new episode ---
        current_state = start_state.copy() # CRITICAL: Start each episode from the exact same state
        total_discounted_reward = 0.0
        current_discount = 1.0

        # 3. Episode Loop: Simulate one full trajectory into the future
        for step in range(EPISODE_LENGTH):
            # a. Choose an agent and get its action from the FIXED policy
            agent_to_act = current_state.get_random_agent_id()
            move_action = nearest_policy(current_state, agent_to_act)
            
            # b. Simulate the full two-step environment transition
            s_half, r_move = update_move_state(current_state, agent_to_act, move_action.vector)
            
            # The reward from the move step (r_move) is always 0, so we only care about the pick reward
            picked = s_half.apples[s_half.agent_position_tuple(agent_to_act)] >= 1
            s_next, r_pick = update_spawn_despawn_and_pick(s_half, agent_to_act, picked)

            # since env is two steps, each applying discount once, we apply discount twice here
            current_discount *= DISCOUNT
            # c. Accumulate the discounted reward for this step
            total_discounted_reward += current_discount * r_pick
            
            # d. Decay the discount factor for the next step
            current_discount *= DISCOUNT
            
            # e. Move to the next state
            current_state = s_next

        # After the episode is finished, store the final calculated value
        value_samples.append(total_discounted_reward)

    # After running all samples, calculate the final estimated value for the start_state
    estimated_value = np.mean(value_samples)
    std_dev = np.std(value_samples)
    true_value_results[start_state.name] = (estimated_value, std_dev)
    
    print(f"\\n>>> Estimated True Value for '{start_state.name}': {estimated_value:.4f} ± {std_dev:.4f}")

\n--- Running Monte Carlo Simulation for state: 'State after 20 steps' ---


Simulating Episodes: 100%|██████████| 1000/1000 [00:38<00:00, 26.25it/s]


\n>>> Estimated True Value for 'State after 20 steps': 28.1882 ± 2.1395
\n--- Running Monte Carlo Simulation for state: 'State after 500 steps' ---


Simulating Episodes: 100%|██████████| 1000/1000 [00:36<00:00, 27.14it/s]


\n>>> Estimated True Value for 'State after 500 steps': 29.1446 ± 2.1813
\n--- Running Monte Carlo Simulation for state: 'State after 999 steps' ---


Simulating Episodes: 100%|██████████| 1000/1000 [00:37<00:00, 26.80it/s]

\n>>> Estimated True Value for 'State after 999 steps': 27.6607 ± 2.1653


In [7]:
print("\\n--- Final Monte Carlo Value Estimates ---")
for name, (mean, std) in true_value_results.items():
    print(f"  - State: '{name}'")
    print(f"    - Estimated Value: {mean:.4f}")
    print(f"    - Standard Deviation: {std:.4f}")

\n--- Final Monte Carlo Value Estimates ---
  - State: 'State after 20 steps'
    - Estimated Value: 28.1882
    - Standard Deviation: 2.1395
  - State: 'State after 500 steps'
    - Estimated Value: 29.1446
    - Standard Deviation: 2.1813
  - State: 'State after 999 steps'
    - Estimated Value: 27.6607
    - Standard Deviation: 2.1653
